# 08 | EIS Degradation Mechanisms and Cross-Modal Health Coupling

## Study objective

This notebook tests whether electrochemical impedance spectroscopy provides physically interpretable information about SOFC degradation and future health beyond the current SOH and assessment index.

The analysis separates four questions:

1. Are the raw spectra sufficiently consistent for comparison?
2. Which EIS quantities change within a cell as degradation progresses?
3. Are those changes associated with IV, transient and composite-health measurements?
4. Do EIS features improve leakage-safe future-SOH forecasts for a completely unseen cell?

The physical cell is the independent engineering unit. Rows from the same cell are repeated measurements, not independent experiments. The horizontal axis is the degradation assessment index, not operating hours or true lifetime.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, theilslopes
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

try:
    from impedance.models.circuits import CustomCircuit
    from pyimpspec import (
        DataSet,
        calculate_drt,
        fit_circuit,
        parse_cdc,
        perform_kramers_kronig_test,
    )
except ImportError as exc:
    raise ImportError(
        "Notebook 08 requires pyimpspec and impedance.py. With the project "
        "environment active, run: python -m pip install "
        '"pyimpspec>=5.1,<6" "impedance>=1.7,<2"'
    ) from exc

try:
    PYIMPSPEC_VERSION = version("pyimpspec")
except PackageNotFoundError:
    PYIMPSPEC_VERSION = "unknown"

try:
    IMPEDANCE_VERSION = version("impedance")
except PackageNotFoundError:
    IMPEDANCE_VERSION = "unknown"

sns.set_theme(style="whitegrid", context="talk")


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate pyproject.toml. Start Jupyter Notebook from inside the project directory."
    )


def first_existing_column(frame: pd.DataFrame, candidates: list[str]) -> str:
    """Return the first matching schema candidate."""
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(
        "None of the expected columns were found: "
        f"{candidates}. Available columns: {frame.columns.tolist()}"
    )


ROOT = find_project_root(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
EIS_PATH = PROCESSED / "eis.parquet"
MODEL_PATH = PROCESSED / "modeling_table.parquet"
UNCERTAINTY_PATH = ROOT / "reports" / "tables" / "uncertainty" / "uncertainty_reliability_audit.csv"
OUTPUT_DIRECTORY = ROOT / "reports" / "tables" / "eis_health_coupling"
FIGURE_DIRECTORY = ROOT / "reports" / "figures" / "eis_health_coupling"

for directory in (OUTPUT_DIRECTORY, FIGURE_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

required_files = [EIS_PATH, MODEL_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    missing_text = "\n".join(f" - {path}" for path in missing_files)
    raise FileNotFoundError(
        f"Notebook 08 requires these files:\n{missing_text}\n\nRun the data pipeline first."
    )

eis = pd.read_parquet(EIS_PATH)
modeling = (
    pd.read_parquet(MODEL_PATH).sort_values(["cell_id", "assessment_index"]).reset_index(drop=True)
)
modeling["regime"] = np.where(
    modeling["cell_id"].str.startswith("R"),
    "randomized",
    "regular",
)

FREQUENCY_COLUMN = first_existing_column(
    eis,
    ["frequency_hz", "eis_frequency_hz", "frequency", "freq_hz"],
)
REAL_COLUMN = first_existing_column(
    eis,
    ["z_real_ohm", "real_z_ohm", "z_real", "real_ohm"],
)
IMAG_COLUMN = first_existing_column(
    eis,
    ["z_imag_ohm", "imag_z_ohm", "z_imag", "imag_ohm"],
)

required_keys = {"cell_id", "assessment_index"}
for frame_name, frame in {"eis": eis, "modeling": modeling}.items():
    missing = required_keys.difference(frame.columns)
    if missing:
        raise KeyError(f"{frame_name} is missing keys: {sorted(missing)}")

EIS_FEATURES = [
    column
    for column in modeling.columns
    if column.startswith("eis_") and pd.api.types.is_numeric_dtype(modeling[column])
]
CORE_EIS_FEATURES = [
    feature
    for feature in [
        "eis_r_ohmic_ohm",
        "eis_polarization_proxy_ohm",
    ]
    if feature in modeling.columns
]
OUTCOMES = [
    outcome
    for outcome in [
        "soh_composite_pct",
        "iv_max_power_w_cm2",
        "iv_current_at_0p70v_a_cm2",
        "tr_performance_current_a_cm2",
    ]
    if outcome in modeling.columns
]

if len(CORE_EIS_FEATURES) < 2:
    raise KeyError(
        "The modeling table must contain eis_r_ohmic_ohm and eis_polarization_proxy_ohm."
    )
if "soh_composite_pct" not in OUTCOMES:
    raise KeyError("soh_composite_pct is required.")

print("Python:", sys.version.split()[0])
print("pyimpspec:", PYIMPSPEC_VERSION)
print("impedance.py:", IMPEDANCE_VERSION)
print("Project root:", ROOT)
print("Raw EIS shape:", eis.shape)
print("Modeling-table shape:", modeling.shape)
print("Cells:", sorted(modeling["cell_id"].unique()))
print("Detected EIS columns:", FREQUENCY_COLUMN, REAL_COLUMN, IMAG_COLUMN)
print("Assessment-level EIS features:", len(EIS_FEATURES))
print("Cross-modal outcomes:", OUTCOMES)

## 1. Data contract and spectral quality control

An impedance spectrum is a set of complex measurements

$$
Z(\omega)=Z'(\omega)+jZ''(\omega)
$$

recorded across angular frequency $\omega=2\pi f$. Before interpreting arcs or fitting equivalent circuits, we verify that spectra have valid frequencies, finite impedance values and comparable acquisition coverage.

Quality control is not cosmetic. Removing a frequency region, reversing the imaginary sign or comparing spectra with different frequency support can change the inferred resistance and mechanism.


In [ ]:
eis = eis.copy()
eis["regime"] = np.where(
    eis["cell_id"].str.startswith("R"),
    "randomized",
    "regular",
)

spectral_audit = (
    eis.groupby(["cell_id", "assessment_index"], observed=True)
    .agg(
        frequency_points=(FREQUENCY_COLUMN, "size"),
        unique_frequencies=(FREQUENCY_COLUMN, "nunique"),
        minimum_frequency_hz=(FREQUENCY_COLUMN, "min"),
        maximum_frequency_hz=(FREQUENCY_COLUMN, "max"),
        missing_frequency=(FREQUENCY_COLUMN, lambda values: int(values.isna().sum())),
        missing_real=(REAL_COLUMN, lambda values: int(values.isna().sum())),
        missing_imag=(IMAG_COLUMN, lambda values: int(values.isna().sum())),
        nonpositive_frequency=(
            FREQUENCY_COLUMN,
            lambda values: int((values <= 0).sum()),
        ),
    )
    .reset_index()
)
spectral_audit["duplicate_frequency_rows"] = (
    spectral_audit["frequency_points"] - spectral_audit["unique_frequencies"]
)
spectral_audit["passes_basic_qc"] = (
    spectral_audit[
        [
            "missing_frequency",
            "missing_real",
            "missing_imag",
            "nonpositive_frequency",
            "duplicate_frequency_rows",
        ]
    ]
    .sum(axis=1)
    .eq(0)
)

coverage_audit = (
    spectral_audit.groupby("cell_id", observed=True)
    .agg(
        spectra=("assessment_index", "nunique"),
        median_frequency_points=("frequency_points", "median"),
        minimum_frequency_points=("frequency_points", "min"),
        maximum_frequency_points=("frequency_points", "max"),
        minimum_frequency_hz=("minimum_frequency_hz", "median"),
        maximum_frequency_hz=("maximum_frequency_hz", "median"),
        qc_pass_rate=("passes_basic_qc", "mean"),
    )
    .reset_index()
)
coverage_audit.insert(
    1,
    "regime",
    np.where(
        coverage_audit["cell_id"].str.startswith("R"),
        "randomized redox",
        "regular",
    ),
)

spectral_audit.to_csv(
    OUTPUT_DIRECTORY / "spectral_quality_audit.csv",
    index=False,
)
coverage_audit.to_csv(
    OUTPUT_DIRECTORY / "spectral_coverage_by_cell.csv",
    index=False,
)

print("Spectra audited:", len(spectral_audit))
print("Basic-QC failures:", int((~spectral_audit["passes_basic_qc"]).sum()))
display(coverage_audit.round(4))

assert len(spectral_audit) == 418, "Expected 418 assessment-level spectra."
assert spectral_audit["passes_basic_qc"].all(), "Review failed structural-QC spectra."

### 1.1 All-spectrum visual audit

The table above establishes structural completeness. The Nyquist panels below show all 418 spectra and expose discontinuities that count-based QC cannot detect.


In [ ]:
plot_frame = (
    eis[
        [
            "cell_id",
            "assessment_index",
            FREQUENCY_COLUMN,
            REAL_COLUMN,
            IMAG_COLUMN,
        ]
    ]
    .dropna()
    .rename(
        columns={
            FREQUENCY_COLUMN: "frequency_hz",
            REAL_COLUMN: "z_real_ohm",
            IMAG_COLUMN: "z_imag_ohm",
        }
    )
    .sort_values(
        ["cell_id", "assessment_index", "frequency_hz"],
        ascending=[True, True, False],
    )
)

ordered_cells = [
    *sorted(cell for cell in plot_frame["cell_id"].unique() if cell.startswith("N")),
    *sorted(cell for cell in plot_frame["cell_id"].unique() if cell.startswith("R")),
]
spectra_per_cell = (
    plot_frame.groupby("cell_id", observed=True)["assessment_index"]
    .nunique()
    .reindex(ordered_cells)
)

assessment_norm = plt.Normalize(
    vmin=float(plot_frame["assessment_index"].min()),
    vmax=float(plot_frame["assessment_index"].max()),
)
assessment_cmap = plt.colormaps["viridis"]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = np.asarray(axes).ravel()
fig.subplots_adjust(
    left=0.07,
    right=0.90,
    bottom=0.08,
    top=0.82,
    wspace=0.38,
    hspace=0.48,
)
fig.suptitle(
    "Evolution of all EIS spectra by physical cell",
    x=0.49,
    y=0.975,
    fontsize=16,
    fontweight="bold",
)
fig.text(
    0.49,
    0.935,
    "Degradation assessment index: chronological diagnostic measurement number for each cell",
    ha="center",
    fontsize=10,
)
fig.text(
    0.49,
    0.905,
    "Index 1 is earliest; higher values are later measurements, not operating hours or degradation magnitude",
    ha="center",
    fontsize=9,
    color="dimgray",
)

for ax, cell_id in zip(axes, ordered_cells, strict=True):
    cell_frame = plot_frame.loc[plot_frame["cell_id"] == cell_id]
    for assessment_index, spectrum in cell_frame.groupby(
        "assessment_index",
        sort=True,
        observed=True,
    ):
        ax.plot(
            spectrum["z_real_ohm"],
            -spectrum["z_imag_ohm"],
            color=assessment_cmap(assessment_norm(float(assessment_index))),
            linewidth=0.85,
            alpha=0.65,
        )
    regime_label = "Regular" if cell_id.startswith("N") else "Randomized redox"
    ax.set_title(
        f"Cell {cell_id}, {regime_label}\n{int(spectra_per_cell.loc[cell_id])} spectra",
        fontsize=10,
        fontweight="bold",
        pad=8,
    )
    ax.set_xlabel(r"$Z^\prime$ ($\Omega$)", fontsize=9)
    ax.set_ylabel(r"$-Z^{\prime\prime}$ ($\Omega$)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.set_aspect("equal", adjustable="datalim")

colour_bar_source = plt.cm.ScalarMappable(
    norm=assessment_norm,
    cmap=assessment_cmap,
)
colour_bar_axis = fig.add_axes([0.925, 0.17, 0.015, 0.55])
colour_bar = fig.colorbar(colour_bar_source, cax=colour_bar_axis)
colour_bar.set_label("Degradation assessment index", fontsize=9, labelpad=10)
colour_bar.ax.tick_params(labelsize=8)

all_spectra_path = FIGURE_DIRECTORY / "all_eis_spectra_by_cell.png"
fig.savefig(all_spectra_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Figure saved to:", all_spectra_path)

### 1.2 Frequency-aware continuity screening

A large adjacent step is not automatically an acquisition failure. The screen compares each frequency transition with the same transition across assessments of the same cell, so repeatable high-frequency geometry is not mislabeled as a random anomaly.


In [ ]:
# A frequency-aware screen distinguishes isolated discontinuities from
# repeatable cell-specific geometry, such as R2's high-frequency response.
CONTINUITY_JUMP_RATIO_LIMIT = 10.0
CONTINUITY_NUMERICAL_TOLERANCE = 1e-3

transition_records = []
for (cell_id, assessment_index), spectrum in plot_frame.groupby(
    ["cell_id", "assessment_index"],
    sort=True,
    observed=True,
):
    spectrum = spectrum.sort_values("frequency_hz", ascending=False).reset_index(drop=True)
    frequency = spectrum["frequency_hz"].to_numpy(dtype=float)
    impedance = spectrum["z_real_ohm"].to_numpy(dtype=float) + 1j * spectrum["z_imag_ohm"].to_numpy(
        dtype=float
    )
    jumps = np.abs(np.diff(impedance))
    for position, jump in enumerate(jumps):
        transition_records.append(
            {
                "cell_id": cell_id,
                "assessment_index": int(assessment_index),
                "frequency_before_hz": float(frequency[position]),
                "frequency_after_hz": float(frequency[position + 1]),
                "adjacent_jump_ohm": float(jump),
            }
        )

transition_audit = pd.DataFrame(transition_records)
transition_reference = (
    transition_audit.groupby(
        ["cell_id", "frequency_before_hz", "frequency_after_hz"],
        observed=True,
    )["adjacent_jump_ohm"]
    .agg(
        transition_median_ohm="median",
        transition_mad_ohm=lambda values: float(np.median(np.abs(values - np.median(values)))),
    )
    .reset_index()
)
transition_audit = transition_audit.merge(
    transition_reference,
    on=["cell_id", "frequency_before_hz", "frequency_after_hz"],
    how="left",
    validate="many_to_one",
)
minimum_scale = np.maximum(
    0.05 * transition_audit["transition_median_ohm"].abs(),
    np.finfo(float).eps,
)
transition_audit["transition_robust_scale_ohm"] = np.maximum(
    1.4826 * transition_audit["transition_mad_ohm"],
    minimum_scale,
)
transition_audit["transition_outlier_score"] = (
    transition_audit["adjacent_jump_ohm"] - transition_audit["transition_median_ohm"]
).abs() / transition_audit["transition_robust_scale_ohm"]

continuity_records = []
for (cell_id, assessment_index), frame in transition_audit.groupby(
    ["cell_id", "assessment_index"],
    sort=True,
    observed=True,
):
    median_jump = float(frame["adjacent_jump_ohm"].median())
    maximum_row = frame.loc[frame["adjacent_jump_ohm"].idxmax()]
    continuity_records.append(
        {
            "cell_id": cell_id,
            "regime": "regular" if cell_id.startswith("N") else "randomized",
            "assessment_index": int(assessment_index),
            "maximum_adjacent_jump_ohm": float(maximum_row["adjacent_jump_ohm"]),
            "maximum_to_median_jump_ratio": (
                float(maximum_row["adjacent_jump_ohm"] / median_jump) if median_jump > 0 else np.nan
            ),
            "maximum_transition_outlier_score": float(frame["transition_outlier_score"].max()),
            "outlying_transition_count": int((frame["transition_outlier_score"] >= 8.0).sum()),
            "frequency_before_jump_hz": float(maximum_row["frequency_before_hz"]),
            "frequency_after_jump_hz": float(maximum_row["frequency_after_hz"]),
        }
    )

continuity_audit = pd.DataFrame(continuity_records)
continuity_audit["continuity_review_required"] = (
    continuity_audit["maximum_to_median_jump_ratio"]
    >= CONTINUITY_JUMP_RATIO_LIMIT - CONTINUITY_NUMERICAL_TOLERANCE
) & (continuity_audit["maximum_transition_outlier_score"] >= 8.0)

QC_CHALLENGE_CASES = {
    "N2": [12, 44, 49],
    "R1": [13],
}
continuity_audit["qc_challenge_case"] = [
    int(assessment) in QC_CHALLENGE_CASES.get(cell_id, [])
    for cell_id, assessment in zip(
        continuity_audit["cell_id"],
        continuity_audit["assessment_index"],
        strict=True,
    )
]

continuity_by_cell = (
    continuity_audit.groupby(["cell_id", "regime"], observed=True)
    .agg(
        spectra_audited=("assessment_index", "count"),
        spectra_flagged=("continuity_review_required", "sum"),
        maximum_jump_ratio=("maximum_to_median_jump_ratio", "max"),
        maximum_transition_outlier_score=(
            "maximum_transition_outlier_score",
            "max",
        ),
    )
    .reset_index()
)
continuity_by_cell["flagged_rate"] = (
    continuity_by_cell["spectra_flagged"] / continuity_by_cell["spectra_audited"]
)

continuity_audit.to_csv(
    OUTPUT_DIRECTORY / "eis_continuity_audit.csv",
    index=False,
)
continuity_by_cell.to_csv(
    OUTPUT_DIRECTORY / "eis_continuity_by_cell.csv",
    index=False,
)

print("Frequency-aware continuity screening")
display(continuity_by_cell.round(4))
print("Pre-specified QC challenge spectra")
display(
    continuity_audit.loc[
        continuity_audit["qc_challenge_case"],
        [
            "cell_id",
            "assessment_index",
            "maximum_adjacent_jump_ohm",
            "maximum_to_median_jump_ratio",
            "maximum_transition_outlier_score",
            "outlying_transition_count",
            "frequency_before_jump_hz",
            "frequency_after_jump_hz",
            "continuity_review_required",
        ],
    ].round(4)
)

## 2. Nyquist and Bode evolution

The Nyquist representation plots $-Z''$ against $Z'$. The high-frequency real-axis intercept approximates ohmic resistance, while the horizontal extent of the low-frequency response provides a polarization-resistance proxy.

The Bode representation retains frequency ordering and is useful when overlapping arcs make the Nyquist plot ambiguous:

$$
|Z|=\sqrt{(Z')^2+(Z'')^2}
$$

$$
\theta=\tan^{-1}\left(\frac{Z''}{Z'}\right)
$$

Visual changes are treated as hypotheses, not proof that a unique electrochemical mechanism has been identified.


In [ ]:
def representative_assessments(frame: pd.DataFrame) -> list[int]:
    """Choose observed early, middle and late assessments."""
    values = np.sort(frame["assessment_index"].dropna().unique())
    positions = np.unique(np.rint(np.linspace(0, len(values) - 1, 3)).astype(int))
    return [int(values[position]) for position in positions]


comparison_cells = [cell for cell in ["N1", "R1"] if cell in eis["cell_id"].unique()]
fig, axes = plt.subplots(
    len(comparison_cells),
    2,
    figsize=(15, 6 * len(comparison_cells)),
    squeeze=False,
)

for row_index, cell_id in enumerate(comparison_cells):
    cell_eis = eis.loc[eis["cell_id"] == cell_id]
    selected = representative_assessments(cell_eis)

    for assessment in selected:
        spectrum = cell_eis.loc[cell_eis["assessment_index"] == assessment].sort_values(
            FREQUENCY_COLUMN, ascending=False
        )
        frequency = spectrum[FREQUENCY_COLUMN].to_numpy()
        real = spectrum[REAL_COLUMN].to_numpy()
        imag = spectrum[IMAG_COLUMN].to_numpy()
        magnitude = np.sqrt(real**2 + imag**2)

        axes[row_index, 0].plot(
            real,
            -imag,
            marker="o",
            markersize=3,
            linewidth=1.5,
            label=f"Assessment {assessment}",
        )
        axes[row_index, 1].plot(
            frequency,
            magnitude,
            marker="o",
            markersize=3,
            linewidth=1.5,
            label=f"Assessment {assessment}",
        )

    axes[row_index, 0].set_title(
        f"Cell {cell_id}: Nyquist evolution",
        fontweight="bold",
    )
    axes[row_index, 0].set_xlabel("Real impedance, Z' (ohm)")
    axes[row_index, 0].set_ylabel("Negative imaginary impedance, -Z'' (ohm)")
    axes[row_index, 0].legend(fontsize=9)

    axes[row_index, 1].set_xscale("log")
    axes[row_index, 1].set_yscale("log")
    axes[row_index, 1].set_title(
        f"Cell {cell_id}: Bode magnitude",
        fontweight="bold",
    )
    axes[row_index, 1].set_xlabel("Frequency (Hz)")
    axes[row_index, 1].set_ylabel("Impedance magnitude, |Z| (ohm)")
    axes[row_index, 1].legend(fontsize=9)

plt.tight_layout()
spectra_figure_path = FIGURE_DIRECTORY / "representative_eis_evolution.png"
plt.savefig(spectra_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", spectra_figure_path)

## 3. Formal impedance validation and cross-package sensitivity

Every one of the 418 spectra is first subjected to the same fixed-complexity Lin-KK test. A spectrum is accepted for EIS interpretation only when both conditions hold:

$$
\mathrm{RMSE}_{\mathrm{Lin-KK}} \leq 1\% 
\qquad	ext{and}\qquad
\max |r_{\mathrm{Lin-KK}}| \leq 5\%.
$$

These are project screening limits, not universal electrochemical standards. They are appropriate here because this is controlled laboratory data with a common 61-point frequency grid, and the preliminary smooth representative spectra produced RMSE values of about 0.179% to 0.366% and maximum residuals of about 0.903% to 1.542%, all well below these limits. The 1% and 5% limits therefore provide practical headroom above the typical laboratory residuals while still rejecting spectra whose lack of Lin-KK consistency could distort circuit or health interpretation. They must be reconsidered for another instrument, frequency grid, noise level or field dataset.

The raw observations are never deleted. Failed spectra remain in the audit outputs, but they are excluded from circuit fitting, DRT interpretation, cross-modal association and predictive modeling.

After the all-spectrum screen, early, middle and late passing spectra from one regular cell and one randomized-redox cell receive three specialist checks:

1. **Equivalent-circuit benchmark:** the same deliberately simple series-resistance plus parallel-RC circuit is fitted independently with both packages. `pyimpspec` expresses it as `R(RC)`, while `impedance.py` expresses the identical topology as `R0-p(R1,C1)`.
2. **Distribution of relaxation times:** the non-negative least-squares DRT method identifies resolved relaxation-time peaks without choosing a multi-arc circuit in advance.
3. **Cross-package sensitivity:** resistance, capacitance and complex-impedance RMSE are compared between implementations.

The fixed 18-RC Lin-KK configuration is a consistent screening model, not a claim that the spectrum contains 18 physical processes. Agreement between packages supports numerical robustness, but does not prove a unique mechanistic interpretation.


In [ ]:
SPECIALIST_CELLS = [cell_id for cell_id in ["N1", "R1"] if cell_id in eis["cell_id"].unique()]
SPECIALIST_ASSESSMENTS_PER_CELL = 3
SPECIALIST_CIRCUIT = "R(RC)"
IMPEDANCE_CIRCUIT = "R0-p(R1,C1)"
SPECIALIST_KK_RC_ELEMENTS = 18


def make_impedance_dataset(
    spectrum: pd.DataFrame,
    label: str,
) -> DataSet:
    """Convert one audited spectrum to a pyimpspec DataSet."""
    clean = (
        spectrum[[FREQUENCY_COLUMN, REAL_COLUMN, IMAG_COLUMN]]
        .dropna()
        .sort_values(FREQUENCY_COLUMN, ascending=False)
        .drop_duplicates(FREQUENCY_COLUMN, keep="first")
    )
    frequency = clean[FREQUENCY_COLUMN].to_numpy(dtype=float)
    impedance = clean[REAL_COLUMN].to_numpy(dtype=float) + 1j * clean[IMAG_COLUMN].to_numpy(
        dtype=float
    )
    if len(frequency) < 8:
        raise ValueError(
            f"{label} has only {len(frequency)} valid unique frequencies. "
            "At least eight are required for the specialist analysis."
        )
    return DataSet(
        frequencies=frequency,
        impedances=impedance,
        label=label,
    )

### 3.1 All-spectrum Lin-KK acceptance audit

The following cell runs the same Lin-KK calculation on every structurally valid spectrum. Pass/fail requires both residual criteria. The summary table reports counts and rates by cell, plus a total row. The two occurrence curves show the empirical distributions of residual RMSE and maximum absolute residual, with the acceptance limits marked explicitly.


In [ ]:
LIN_KK_RMSE_LIMIT_PCT = 1.0
LIN_KK_MAX_RESIDUAL_LIMIT_PCT = 5.0

# Preserve complete raw tables. Exclusion below applies only to downstream
# interpretation and modeling, never to the source observations.
eis_all = eis.copy()
modeling_all = modeling.copy()
all_kk_records = []
all_kk_results = {}

for (cell_id, assessment_index), spectrum in eis_all.groupby(
    ["cell_id", "assessment_index"],
    sort=True,
    observed=True,
):
    label = f"{cell_id}_assessment_{int(assessment_index)}"
    keys = {
        "cell_id": cell_id,
        "regime": "randomized" if cell_id.startswith("R") else "regular",
        "assessment_index": int(assessment_index),
        "label": label,
    }
    try:
        data = make_impedance_dataset(spectrum, label)
        kk_result = perform_kramers_kronig_test(
            data,
            num_RC=SPECIALIST_KK_RC_ELEMENTS,
            admittance=False,
            num_F_ext_evaluations=0,
            log_F_ext=0.0,
        )
        _, real_residual_pct, imag_residual_pct = kk_result.get_residuals_data()
        residuals_pct = np.r_[real_residual_pct, imag_residual_pct]
        residual_rmse_pct = float(np.sqrt(np.mean(residuals_pct**2)))
        maximum_residual_pct = float(np.max(np.abs(residuals_pct)))
        passes_lin_kk = bool(
            residual_rmse_pct <= LIN_KK_RMSE_LIMIT_PCT
            and maximum_residual_pct <= LIN_KK_MAX_RESIDUAL_LIMIT_PCT
        )
        all_kk_records.append(
            {
                **keys,
                "status": "success",
                "num_rc": int(kk_result.get_num_RC()),
                "estimated_noise_pct": float(kk_result.get_estimated_percent_noise()),
                "residual_rmse_pct": residual_rmse_pct,
                "maximum_absolute_residual_pct": maximum_residual_pct,
                "passes_rmse_limit": (residual_rmse_pct <= LIN_KK_RMSE_LIMIT_PCT),
                "passes_maximum_residual_limit": (
                    maximum_residual_pct <= LIN_KK_MAX_RESIDUAL_LIMIT_PCT
                ),
                "passes_lin_kk": passes_lin_kk,
                "interpretation_action": (
                    "retain_for_eis_interpretation"
                    if passes_lin_kk
                    else "exclude_from_eis_interpretation"
                ),
                "error": "",
            }
        )
        all_kk_results[(cell_id, int(assessment_index))] = {
            "dataset": data,
            "kk": kk_result,
        }
    except Exception as exc:
        all_kk_records.append(
            {
                **keys,
                "status": "failed",
                "num_rc": np.nan,
                "estimated_noise_pct": np.nan,
                "residual_rmse_pct": np.nan,
                "maximum_absolute_residual_pct": np.nan,
                "passes_rmse_limit": False,
                "passes_maximum_residual_limit": False,
                "passes_lin_kk": False,
                "interpretation_action": "exclude_from_eis_interpretation",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

all_spectrum_lin_kk = pd.DataFrame(all_kk_records).merge(
    continuity_audit[
        [
            "cell_id",
            "assessment_index",
            "continuity_review_required",
            "maximum_to_median_jump_ratio",
            "maximum_transition_outlier_score",
        ]
    ],
    on=["cell_id", "assessment_index"],
    how="left",
    validate="one_to_one",
)

lin_kk_by_cell = (
    all_spectrum_lin_kk.groupby(["cell_id", "regime"], observed=True)
    .agg(
        spectra_tested=("assessment_index", "size"),
        spectra_passed=("passes_lin_kk", "sum"),
        calculation_failures=("status", lambda values: int((values != "success").sum())),
        median_rmse_pct=("residual_rmse_pct", "median"),
        maximum_rmse_pct=("residual_rmse_pct", "max"),
        median_maximum_residual_pct=("maximum_absolute_residual_pct", "median"),
        maximum_residual_pct=("maximum_absolute_residual_pct", "max"),
    )
    .reset_index()
)
lin_kk_by_cell["spectra_failed"] = (
    lin_kk_by_cell["spectra_tested"] - lin_kk_by_cell["spectra_passed"]
)
lin_kk_by_cell["pass_rate"] = lin_kk_by_cell["spectra_passed"] / lin_kk_by_cell["spectra_tested"]
total_row = pd.DataFrame(
    [
        {
            "cell_id": "TOTAL",
            "regime": "all",
            "spectra_tested": len(all_spectrum_lin_kk),
            "spectra_passed": int(all_spectrum_lin_kk["passes_lin_kk"].sum()),
            "calculation_failures": int((all_spectrum_lin_kk["status"] != "success").sum()),
            "median_rmse_pct": all_spectrum_lin_kk["residual_rmse_pct"].median(),
            "maximum_rmse_pct": all_spectrum_lin_kk["residual_rmse_pct"].max(),
            "median_maximum_residual_pct": all_spectrum_lin_kk[
                "maximum_absolute_residual_pct"
            ].median(),
            "maximum_residual_pct": all_spectrum_lin_kk["maximum_absolute_residual_pct"].max(),
            "spectra_failed": int((~all_spectrum_lin_kk["passes_lin_kk"]).sum()),
            "pass_rate": float(all_spectrum_lin_kk["passes_lin_kk"].mean()),
        }
    ]
)
lin_kk_pass_fail_table = pd.concat(
    [lin_kk_by_cell, total_row],
    ignore_index=True,
)

all_spectrum_lin_kk.to_csv(
    OUTPUT_DIRECTORY / "all_spectrum_lin_kk_audit.csv",
    index=False,
)
lin_kk_pass_fail_table.to_csv(
    OUTPUT_DIRECTORY / "lin_kk_pass_fail_by_cell.csv",
    index=False,
)

print("All-spectrum Lin-KK acceptance audit")
print("RMSE acceptance limit:", f"{LIN_KK_RMSE_LIMIT_PCT:.1f}%")
print(
    "Maximum absolute residual acceptance limit:",
    f"{LIN_KK_MAX_RESIDUAL_LIMIT_PCT:.1f}%",
)
display(lin_kk_pass_fail_table.round(4))

successful_kk = all_spectrum_lin_kk.loc[all_spectrum_lin_kk["status"] == "success"]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
error_specs = [
    (
        "residual_rmse_pct",
        LIN_KK_RMSE_LIMIT_PCT,
        "Lin-KK residual RMSE (%)",
        "Residual RMSE occurrence",
    ),
    (
        "maximum_absolute_residual_pct",
        LIN_KK_MAX_RESIDUAL_LIMIT_PCT,
        "Maximum absolute Lin-KK residual (%)",
        "Maximum-residual occurrence",
    ),
]
for axis, (column, limit, xlabel, title) in zip(
    axes,
    error_specs,
    strict=True,
):
    values = successful_kk[column].dropna().to_numpy(dtype=float)
    positive_values = values[values > 0]
    bin_edges = np.geomspace(
        positive_values.min() * 0.95,
        positive_values.max() * 1.05,
        36,
    )
    counts, edges = np.histogram(positive_values, bins=bin_edges)
    centers = np.sqrt(edges[:-1] * edges[1:])
    observed_bins = counts > 0
    axis.plot(
        centers[observed_bins],
        counts[observed_bins],
        marker="o",
        linewidth=2,
    )
    axis.fill_between(
        centers[observed_bins],
        counts[observed_bins],
        1,
        alpha=0.2,
    )
    axis.axvline(
        limit,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Acceptance limit = {limit:.1f}%",
    )
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel(xlabel)
    axis.set_ylabel("Number of spectra (log scale)")
    axis.set_title(title, fontweight="bold")
    axis.legend(fontsize=9)

fig.suptitle(
    "Lin-KK error occurrence across all EIS spectra",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
lin_kk_error_figure = FIGURE_DIRECTORY / "all_spectrum_lin_kk_error_occurrence.png"
fig.savefig(lin_kk_error_figure, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", lin_kk_error_figure)

failed_lin_kk_spectra = (
    all_spectrum_lin_kk.loc[
        ~all_spectrum_lin_kk["passes_lin_kk"],
        [
            "cell_id",
            "regime",
            "assessment_index",
            "residual_rmse_pct",
            "maximum_absolute_residual_pct",
            "passes_rmse_limit",
            "passes_maximum_residual_limit",
            "continuity_review_required",
            "maximum_to_median_jump_ratio",
            "maximum_transition_outlier_score",
            "interpretation_action",
            "error",
        ],
    ]
    .sort_values(["cell_id", "assessment_index"])
    .reset_index(drop=True)
)
failed_lin_kk_spectra.to_csv(
    OUTPUT_DIRECTORY / "failed_lin_kk_spectra.csv",
    index=False,
)
print("Spectra failing the Lin-KK acceptance screen:", len(failed_lin_kk_spectra))
display(failed_lin_kk_spectra.round(4))

# Downstream EIS analysis uses only spectra that pass both Lin-KK limits.
accepted_keys = all_spectrum_lin_kk.loc[
    all_spectrum_lin_kk["passes_lin_kk"],
    ["cell_id", "assessment_index"],
]
eis = eis_all.merge(
    accepted_keys,
    on=["cell_id", "assessment_index"],
    how="inner",
    validate="many_to_one",
)
modeling = modeling_all.merge(
    accepted_keys,
    on=["cell_id", "assessment_index"],
    how="inner",
    validate="one_to_one",
)
print("Spectra retained for downstream EIS interpretation:", len(accepted_keys))
print(
    "Spectra excluded from downstream EIS interpretation:",
    int((~all_spectrum_lin_kk["passes_lin_kk"]).sum()),
)

In [ ]:
specialist_spectra = []

for cell_id in SPECIALIST_CELLS:
    raw_cell_frame = eis_all.loc[eis_all["cell_id"] == cell_id]
    accepted_cell_frame = eis.loc[eis["cell_id"] == cell_id]

    target_assessments = representative_assessments(raw_cell_frame)
    accepted_assessments = np.sort(accepted_cell_frame["assessment_index"].unique())

    selected_assessments = []
    for target in target_assessments:
        nearest_position = np.argmin(np.abs(accepted_assessments - target))
        selected_assessments.append(int(accepted_assessments[nearest_position]))

    selected_assessments = list(dict.fromkeys(selected_assessments))

    for assessment_index in selected_assessments:
        spectrum = accepted_cell_frame.loc[
            accepted_cell_frame["assessment_index"] == assessment_index
        ]

        label = f"{cell_id}_assessment_{assessment_index}"

        specialist_spectra.append(
            {
                "cell_id": cell_id,
                "regime": ("randomized" if cell_id.startswith("R") else "regular"),
                "assessment_index": assessment_index,
                "label": label,
                "dataset": make_impedance_dataset(
                    spectrum,
                    label,
                ),
            }
        )

specialist_selection_table = pd.DataFrame(specialist_spectra).drop(columns="dataset")

print("pyimpspec version:", PYIMPSPEC_VERSION)
print("impedance.py version:", IMPEDANCE_VERSION)
print(
    "Specialist spectra selected:",
    len(specialist_spectra),
)
print(
    "Equivalent circuit:",
    SPECIALIST_CIRCUIT,
    "=",
    IMPEDANCE_CIRCUIT,
)
print(
    "Lin-KK RC elements:",
    SPECIALIST_KK_RC_ELEMENTS,
)
display(specialist_selection_table)

In [ ]:
kk_records = []
ecm_parameter_frames = []
ecm_statistic_frames = []
circuit_comparison_records = []
drt_records = []
specialist_results = {}


def relative_complex_rmse_pct(
    measured: np.ndarray,
    predicted: np.ndarray,
) -> float:
    """Complex RMSE normalized by the RMS magnitude of measured Z."""
    numerator = np.sqrt(np.mean(np.abs(measured - predicted) ** 2))
    denominator = np.sqrt(np.mean(np.abs(measured) ** 2))
    return float(100 * numerator / denominator)


for item in specialist_spectra:
    keys = {
        "cell_id": item["cell_id"],
        "regime": item["regime"],
        "assessment_index": item["assessment_index"],
        "label": item["label"],
    }
    data = item["dataset"]
    result_bundle = {}
    frequency = data.get_frequencies()
    measured_impedance = data.get_impedances()

    initial_feature_row = modeling.loc[
        (modeling["cell_id"] == item["cell_id"])
        & (modeling["assessment_index"] == item["assessment_index"])
    ].iloc[0]
    tiny = np.finfo(float).eps
    initial_r0 = max(
        float(initial_feature_row["eis_r_ohmic_ohm"]),
        tiny,
    )
    initial_r1 = max(
        float(initial_feature_row["eis_polarization_proxy_ohm"]),
        tiny,
    )
    peak_frequency = float(frequency[np.argmax(np.abs(measured_impedance.imag))])
    initial_c1 = 1.0 / (2 * np.pi * peak_frequency * initial_r1)

    try:
        kk_result = perform_kramers_kronig_test(
            data,
            num_RC=SPECIALIST_KK_RC_ELEMENTS,
            admittance=False,
            num_F_ext_evaluations=0,
            log_F_ext=0.0,
        )
        _, kk_real_residual_pct, kk_imag_residual_pct = kk_result.get_residuals_data()
        kk_records.append(
            {
                **keys,
                "status": "success",
                "num_rc": kk_result.get_num_RC(),
                "estimated_noise_pct": (kk_result.get_estimated_percent_noise()),
                "residual_rmse_pct": float(
                    np.sqrt(
                        np.mean(
                            np.r_[
                                kk_real_residual_pct,
                                kk_imag_residual_pct,
                            ]
                            ** 2
                        )
                    )
                ),
                "maximum_absolute_residual_pct": float(
                    np.max(
                        np.abs(
                            np.r_[
                                kk_real_residual_pct,
                                kk_imag_residual_pct,
                            ]
                        )
                    )
                ),
                "error": "",
            }
        )
        result_bundle["kk"] = kk_result
    except Exception as exc:
        kk_records.append(
            {
                **keys,
                "status": "failed",
                "num_rc": np.nan,
                "estimated_noise_pct": np.nan,
                "residual_rmse_pct": np.nan,
                "maximum_absolute_residual_pct": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

    try:
        pyimpspec_circuit = parse_cdc(SPECIALIST_CIRCUIT)
        circuit_elements = pyimpspec_circuit.get_elements()
        circuit_elements[0].set_values(R=initial_r0)
        circuit_elements[1].set_values(R=initial_r1)
        circuit_elements[2].set_values(C=initial_c1)
        fit_result = fit_circuit(
            pyimpspec_circuit,
            data,
            method="least_squares",
            weight="unity",
            max_nfev=100000,
            num_procs=1,
            timeout=0,
        )
        parameter_frame = fit_result.to_parameters_dataframe().copy()
        parameter_frame.insert(0, "label", item["label"])
        parameter_frame.insert(0, "assessment_index", item["assessment_index"])
        parameter_frame.insert(0, "regime", item["regime"])
        parameter_frame.insert(0, "cell_id", item["cell_id"])
        ecm_parameter_frames.append(parameter_frame)

        statistic_frame = fit_result.to_statistics_dataframe().copy()
        statistic_frame.insert(0, "label", item["label"])
        statistic_frame.insert(0, "assessment_index", item["assessment_index"])
        statistic_frame.insert(0, "regime", item["regime"])
        statistic_frame.insert(0, "cell_id", item["cell_id"])
        ecm_statistic_frames.append(statistic_frame)
        result_bundle["ecm"] = fit_result

        fitted_parameters = fit_result.get_parameters()
        fitted_impedance = fit_result.get_impedances()
        circuit_comparison_records.append(
            {
                **keys,
                "package": "pyimpspec",
                "circuit": SPECIALIST_CIRCUIT,
                "r0_ohm": fitted_parameters["R_1"]["R"].value,
                "r1_ohm": fitted_parameters["R_2"]["R"].value,
                "c1_f": fitted_parameters["C_1"]["C"].value,
                "relative_complex_rmse_pct": (
                    relative_complex_rmse_pct(
                        measured_impedance,
                        fitted_impedance,
                    )
                ),
                "status": "success",
                "error": "",
            }
        )
    except Exception as exc:
        ecm_parameter_frames.append(
            pd.DataFrame([{**keys, "error": f"{type(exc).__name__}: {exc}"}])
        )
        circuit_comparison_records.append(
            {
                **keys,
                "package": "pyimpspec",
                "circuit": SPECIALIST_CIRCUIT,
                "r0_ohm": np.nan,
                "r1_ohm": np.nan,
                "c1_f": np.nan,
                "relative_complex_rmse_pct": np.nan,
                "status": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

    try:
        impedance_model = CustomCircuit(
            circuit=IMPEDANCE_CIRCUIT,
            initial_guess=[initial_r0, initial_r1, initial_c1],
        )
        impedance_model.fit(
            frequency,
            measured_impedance,
            weight_by_modulus=False,
            maxfev=100000,
        )
        impedance_prediction = impedance_model.predict(frequency)
        impedance_r0, impedance_r1, impedance_c1 = impedance_model.parameters_
        circuit_comparison_records.append(
            {
                **keys,
                "package": "impedance.py",
                "circuit": IMPEDANCE_CIRCUIT,
                "r0_ohm": impedance_r0,
                "r1_ohm": impedance_r1,
                "c1_f": impedance_c1,
                "relative_complex_rmse_pct": (
                    relative_complex_rmse_pct(
                        measured_impedance,
                        impedance_prediction,
                    )
                ),
                "status": "success",
                "error": "",
            }
        )
        result_bundle["impedance"] = {
            "model": impedance_model,
            "prediction": impedance_prediction,
        }
    except Exception as exc:
        circuit_comparison_records.append(
            {
                **keys,
                "package": "impedance.py",
                "circuit": IMPEDANCE_CIRCUIT,
                "r0_ohm": np.nan,
                "r1_ohm": np.nan,
                "c1_f": np.nan,
                "relative_complex_rmse_pct": np.nan,
                "status": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

    try:
        drt_result = calculate_drt(data, method="tr-nnls")
        peak_tau, peak_gamma = drt_result.get_peaks(threshold=0.05)
        tau = drt_result.get_time_constants()
        gamma = drt_result.get_gammas()
        drt_records.append(
            {
                **keys,
                "status": "success",
                "peak_count": len(peak_tau),
                "dominant_tau_s": (
                    float(peak_tau[np.argmax(peak_gamma)]) if len(peak_tau) else np.nan
                ),
                "dominant_gamma_ohm": (float(np.max(peak_gamma)) if len(peak_gamma) else np.nan),
                "total_intensity_ohm": float(np.trapezoid(gamma, x=np.log(tau))),
                "error": "",
            }
        )
        result_bundle["drt"] = drt_result
    except Exception as exc:
        drt_records.append(
            {
                **keys,
                "status": "failed",
                "peak_count": np.nan,
                "dominant_tau_s": np.nan,
                "dominant_gamma_ohm": np.nan,
                "total_intensity_ohm": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

    specialist_results[item["label"]] = result_bundle

kk_summary = pd.DataFrame(kk_records)
ecm_parameters = pd.concat(
    ecm_parameter_frames,
    ignore_index=True,
    sort=False,
)
ecm_statistics = (
    pd.concat(ecm_statistic_frames, ignore_index=True, sort=False)
    if ecm_statistic_frames
    else pd.DataFrame()
)
drt_summary = pd.DataFrame(drt_records)
circuit_comparison = pd.DataFrame(circuit_comparison_records)

successful_circuit_comparison = circuit_comparison.loc[circuit_comparison["status"] == "success"]
comparison_wide = successful_circuit_comparison.pivot(
    index=["cell_id", "regime", "assessment_index", "label"],
    columns="package",
    values=[
        "r0_ohm",
        "r1_ohm",
        "c1_f",
        "relative_complex_rmse_pct",
    ],
)
comparison_wide.columns = [
    f"{measurement}_{package}" for measurement, package in comparison_wide.columns
]
comparison_wide = comparison_wide.reset_index()

for parameter in ["r0_ohm", "r1_ohm", "c1_f"]:
    pyimpspec_column = f"{parameter}_pyimpspec"
    impedance_column = f"{parameter}_impedance.py"
    if {
        pyimpspec_column,
        impedance_column,
    }.issubset(comparison_wide.columns):
        comparison_wide[f"{parameter}_difference_pct"] = 100 * (
            comparison_wide[impedance_column] / comparison_wide[pyimpspec_column] - 1
        )

kk_summary.to_csv(
    OUTPUT_DIRECTORY / "pyimpspec_kramers_kronig_summary.csv",
    index=False,
)
ecm_parameters.to_csv(
    OUTPUT_DIRECTORY / "pyimpspec_ecm_parameters.csv",
    index=False,
)
ecm_statistics.to_csv(
    OUTPUT_DIRECTORY / "pyimpspec_ecm_statistics.csv",
    index=False,
)
drt_summary.to_csv(
    OUTPUT_DIRECTORY / "pyimpspec_drt_summary.csv",
    index=False,
)
circuit_comparison.to_csv(
    OUTPUT_DIRECTORY / "cross_package_circuit_fits.csv",
    index=False,
)
comparison_wide.to_csv(
    OUTPUT_DIRECTORY / "cross_package_parameter_sensitivity.csv",
    index=False,
)

print("Lin-KK summary")
display(kk_summary.round(6))
print("DRT summary")
display(drt_summary.round(6))
print("Equivalent-circuit parameters")
display(ecm_parameters.round(6))
print("Cross-package circuit comparison")
display(comparison_wide.round(6))

In [ ]:
successful_labels = [
    label
    for label, results in specialist_results.items()
    if {"kk", "ecm", "drt", "impedance"}.issubset(results)
]

if successful_labels:
    fig, axes = plt.subplots(
        len(successful_labels),
        3,
        figsize=(18, 4.5 * len(successful_labels)),
        squeeze=False,
    )

    for row_index, label in enumerate(successful_labels):
        item = next(spectrum for spectrum in specialist_spectra if spectrum["label"] == label)
        data = item["dataset"]
        kk_result = specialist_results[label]["kk"]
        fit_result = specialist_results[label]["ecm"]
        drt_result = specialist_results[label]["drt"]
        impedance_prediction = specialist_results[label]["impedance"]["prediction"]

        measured_real, measured_negative_imag = data.get_nyquist_data()
        kk_real, kk_negative_imag = kk_result.get_nyquist_data()
        fit_real, fit_negative_imag = fit_result.get_nyquist_data()

        axes[row_index, 0].plot(
            measured_real,
            measured_negative_imag,
            "o",
            markersize=4,
            label="Measured",
        )
        axes[row_index, 0].plot(
            kk_real,
            kk_negative_imag,
            linewidth=2,
            label="Lin-KK",
        )
        axes[row_index, 0].plot(
            fit_real,
            fit_negative_imag,
            linewidth=2,
            label="pyimpspec fit",
        )
        axes[row_index, 0].plot(
            impedance_prediction.real,
            -impedance_prediction.imag,
            linestyle="--",
            linewidth=2,
            label="impedance.py fit",
        )
        axes[row_index, 0].set_xlabel("Z' (ohm)")
        axes[row_index, 0].set_ylabel("-Z'' (ohm)")
        axes[row_index, 0].set_title(label, fontweight="bold")
        axes[row_index, 0].legend(fontsize=8)

        residual_frequency, residual_real, residual_imag = kk_result.get_residuals_data()
        axes[row_index, 1].semilogx(
            residual_frequency,
            residual_real,
            label="Real residual",
        )
        axes[row_index, 1].semilogx(
            residual_frequency,
            residual_imag,
            label="Imaginary residual",
        )
        axes[row_index, 1].axhline(
            0,
            color="black",
            linestyle="--",
            linewidth=1,
        )
        axes[row_index, 1].set_xlabel("Frequency (Hz)")
        axes[row_index, 1].set_ylabel("Lin-KK residual (% of |Z|)")
        axes[row_index, 1].legend(fontsize=8)

        axes[row_index, 2].semilogx(
            drt_result.get_time_constants(),
            drt_result.get_gammas(),
            linewidth=2,
        )
        axes[row_index, 2].set_xlabel("Relaxation time, tau (s)")
        axes[row_index, 2].set_ylabel("DRT intensity, gamma (ohm)")
        axes[row_index, 2].set_title("TR-NNLS DRT", fontweight="bold")

    fig.suptitle(
        "Representative validation and cross-package diagnostics",
        fontsize=17,
        fontweight="bold",
    )
    plt.tight_layout()
    specialist_figure_path = FIGURE_DIRECTORY / "pyimpspec_representative_diagnostics.png"
    plt.savefig(
        specialist_figure_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    print("Figure saved to:", specialist_figure_path)
else:
    print(
        "No spectrum completed all three specialist analyses. "
        "Inspect the error columns in the saved summaries."
    )

### 3.2 Detailed review of pre-specified QC challenge spectra

The four visually identified challenge spectra are looked up in the all-spectrum Lin-KK audit. Their raw data remain available for diagnostic plots even when they fail acceptance. No Lin-KK calculation is repeated here, and no failed spectrum proceeds to circuit, DRT, association or forecasting interpretation.


In [ ]:
challenge_mask = all_spectrum_lin_kk.apply(
    lambda row: int(row["assessment_index"]) in QC_CHALLENGE_CASES.get(row["cell_id"], []),
    axis=1,
)
qc_challenge_summary = all_spectrum_lin_kk.loc[challenge_mask].copy()
qc_challenge_summary.to_csv(
    OUTPUT_DIRECTORY / "qc_challenge_lin_kk_summary.csv",
    index=False,
)

print("QC challenge results from the all-spectrum Lin-KK audit")
display(
    qc_challenge_summary[
        [
            "cell_id",
            "assessment_index",
            "residual_rmse_pct",
            "maximum_absolute_residual_pct",
            "passes_lin_kk",
            "interpretation_action",
            "maximum_to_median_jump_ratio",
            "maximum_transition_outlier_score",
        ]
    ].round(5)
)

qc_challenge_results = {
    f"{cell_id}_assessment_{assessment_index}_qc": all_kk_results[(cell_id, int(assessment_index))]
    for cell_id, assessments in QC_CHALLENGE_CASES.items()
    for assessment_index in assessments
    if (cell_id, int(assessment_index)) in all_kk_results
}

if qc_challenge_results:
    fig, axes = plt.subplots(
        len(qc_challenge_results),
        2,
        figsize=(14, 4.2 * len(qc_challenge_results)),
        squeeze=False,
    )
    for row_index, (label, result) in enumerate(qc_challenge_results.items()):
        data = result["dataset"]
        kk_result = result["kk"]
        measured_real, measured_negative_imag = data.get_nyquist_data()
        kk_real, kk_negative_imag = kk_result.get_nyquist_data()
        axes[row_index, 0].plot(
            measured_real,
            measured_negative_imag,
            "o-",
            markersize=3,
            linewidth=1,
            label="Measured",
        )
        axes[row_index, 0].plot(
            kk_real,
            kk_negative_imag,
            linewidth=2,
            label="Lin-KK",
        )
        axes[row_index, 0].set_title(label, fontweight="bold")
        axes[row_index, 0].set_xlabel("Z' (ohm)")
        axes[row_index, 0].set_ylabel("-Z'' (ohm)")
        axes[row_index, 0].legend(fontsize=8)
        residual_frequency, residual_real, residual_imag = kk_result.get_residuals_data()
        axes[row_index, 1].semilogx(
            residual_frequency,
            residual_real,
            label="Real residual",
        )
        axes[row_index, 1].semilogx(
            residual_frequency,
            residual_imag,
            label="Imaginary residual",
        )
        axes[row_index, 1].axhline(0, color="black", linestyle="--", linewidth=1)
        axes[row_index, 1].axhline(
            LIN_KK_MAX_RESIDUAL_LIMIT_PCT,
            color="red",
            linestyle=":",
            linewidth=1,
        )
        axes[row_index, 1].axhline(
            -LIN_KK_MAX_RESIDUAL_LIMIT_PCT,
            color="red",
            linestyle=":",
            linewidth=1,
        )
        axes[row_index, 1].set_xlabel("Frequency (Hz)")
        axes[row_index, 1].set_ylabel("Lin-KK residual (% of |Z|)")
        axes[row_index, 1].legend(fontsize=8)

    fig.suptitle(
        "Physical-consistency review of QC challenge spectra",
        fontsize=16,
        fontweight="bold",
    )
    plt.tight_layout()
    challenge_figure_path = FIGURE_DIRECTORY / "qc_challenge_lin_kk.png"
    fig.savefig(challenge_figure_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Figure saved to:", challenge_figure_path)

## 4. Baseline-relative EIS trajectories

Absolute impedance differs across cells because of manufacturing, geometry, contacts and initial condition. To study degradation within a cell, feature $x$ is expressed relative to its first diagnostic:

$$
\Delta x_{c,k}^{rel}
=
100\left(\frac{x_{c,k}}{x_{c,1}}-1\right)
$$

A positive relative change in resistance means the resistance increased from baseline. Relative normalization supports comparison, but it assumes that the first diagnostic is reliable and available at deployment.


In [ ]:
analysis = modeling.copy()
relative_features = []

for feature in EIS_FEATURES:
    baseline = analysis.groupby("cell_id", observed=True)[feature].transform("first")
    valid_baseline = baseline.notna() & baseline.abs().gt(1e-12)
    relative_name = f"{feature}_relative_pct"
    analysis[relative_name] = np.where(
        valid_baseline,
        100 * (analysis[feature] / baseline - 1),
        np.nan,
    )
    relative_features.append(relative_name)

core_relative_features = [f"{feature}_relative_pct" for feature in CORE_EIS_FEATURES]
initial_deviation = (
    analysis.groupby("cell_id", observed=True)[core_relative_features].first().abs().max().max()
)
if initial_deviation > 1e-8:
    raise ValueError("Baseline-relative EIS features do not start at zero.")

relative_summary = (
    analysis.groupby(["regime", "cell_id"], observed=True)
    .agg(
        assessments=("assessment_index", "nunique"),
        final_r_ohmic_relative_pct=(
            "eis_r_ohmic_ohm_relative_pct",
            "last",
        ),
        final_polarization_relative_pct=(
            "eis_polarization_proxy_ohm_relative_pct",
            "last",
        ),
        final_soh_pct=("soh_composite_pct", "last"),
    )
    .reset_index()
)
relative_summary.to_csv(
    OUTPUT_DIRECTORY / "baseline_relative_eis_summary.csv",
    index=False,
)

print("Baseline-relative EIS features:", len(relative_features))
print("Maximum initial deviation:", float(initial_deviation))
display(relative_summary.round(3))

In [ ]:
trajectory_features = [
    "eis_r_ohmic_ohm_relative_pct",
    "eis_polarization_proxy_ohm_relative_pct",
]

trajectory_titles = {
    "eis_r_ohmic_ohm_relative_pct": "Ohmic resistance",
    "eis_polarization_proxy_ohm_relative_pct": ("Polarization resistance proxy"),
}

cell_order = sorted(modeling_all["cell_id"].unique())
cell_colors = dict(
    zip(
        cell_order,
        sns.color_palette("tab10", len(cell_order)),
        strict=True,
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(17, 6),
    sharex=False,
)

for axis, feature in zip(
    axes,
    trajectory_features,
    strict=True,
):
    for cell_id in cell_order:
        raw_cell = modeling_all.loc[
            modeling_all["cell_id"] == cell_id,
            ["assessment_index"],
        ].drop_duplicates()

        accepted_cell = analysis.loc[
            analysis["cell_id"] == cell_id,
            ["assessment_index", feature],
        ]

        plot_cell = raw_cell.merge(
            accepted_cell,
            on="assessment_index",
            how="left",
            validate="one_to_one",
        ).sort_values("assessment_index")

        axis.plot(
            plot_cell["assessment_index"],
            plot_cell[feature],
            marker="o",
            markersize=2.5,
            linewidth=1.5,
            color=cell_colors[cell_id],
            label=cell_id,
        )

    axis.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )
    axis.set_title(
        trajectory_titles[feature],
        fontweight="bold",
    )
    axis.set_xlabel("Degradation assessment index")
    axis.set_ylabel("Change from initial diagnostic (%)")

axes[1].legend(
    title="Cell ID",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

fig.suptitle(
    "Within-cell evolution of Lin-KK-accepted EIS indicators",
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()

trajectory_figure_path = FIGURE_DIRECTORY / "relative_eis_trajectories.png"

fig.savefig(
    trajectory_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Figure saved to:", trajectory_figure_path)

### 4.1 Cross-modal context for continuity-review cases

Within-cell standardization places SOH and EIS indicators on a common visual scale. Vertical markers identify the pre-specified N2 and R1 challenge assessments.


In [ ]:
# This section deliberately uses the complete modeling table, including
# Lin-KK-rejected spectra, only to diagnose whether EIS anomalies coincide
# with changes in SOH or other EIS indicators.
#
# These rejected observations remain excluded from circuit interpretation,
# association analysis and forecasting.

diagnostic_analysis = modeling_all.copy()

diagnostic_analysis["regime"] = np.where(
    diagnostic_analysis["cell_id"].str.startswith("R"),
    "randomized",
    "regular",
)

# Calculate baseline-relative EIS indicators using each cell's original
# first assessment. This preserves the original diagnostic reference.
for feature in CORE_EIS_FEATURES:
    baseline = diagnostic_analysis.groupby(
        "cell_id",
        observed=True,
    )[feature].transform("first")

    relative_name = f"{feature}_relative_pct"

    diagnostic_analysis[relative_name] = np.where(
        baseline.notna() & baseline.abs().gt(1e-12),
        100 * (diagnostic_analysis[feature] / baseline - 1),
        np.nan,
    )

qc_alignment_signals = [
    "soh_composite_pct",
    "eis_r_ohmic_ohm_relative_pct",
    "eis_polarization_proxy_ohm_relative_pct",
]

signal_names = {
    "soh_composite_pct": "Composite SOH",
    "eis_r_ohmic_ohm_relative_pct": ("Ohmic resistance"),
    "eis_polarization_proxy_ohm_relative_pct": ("Polarization resistance"),
}

qc_cells = ["N2", "R1"]

fig, axes = plt.subplots(
    len(qc_cells),
    1,
    figsize=(14, 10),
    sharex=False,
    squeeze=False,
)

axes = axes.flatten()
alignment_records = []

for axis, cell_id in zip(
    axes,
    qc_cells,
    strict=True,
):
    cell_frame = (
        diagnostic_analysis.loc[
            diagnostic_analysis["cell_id"] == cell_id,
            [
                "assessment_index",
                *qc_alignment_signals,
            ],
        ]
        .sort_values("assessment_index")
        .copy()
    )

    standardized_columns = []

    for signal in qc_alignment_signals:
        valid_values = cell_frame[signal].dropna()

        signal_mean = valid_values.mean()
        signal_scale = valid_values.std(ddof=0)

        standardized_name = f"{signal}_z"

        if np.isfinite(signal_scale) and signal_scale > 0:
            cell_frame[standardized_name] = (cell_frame[signal] - signal_mean) / signal_scale
        else:
            cell_frame[standardized_name] = np.nan

        standardized_columns.append(standardized_name)

    long_frame = cell_frame.melt(
        id_vars="assessment_index",
        value_vars=standardized_columns,
        var_name="signal",
        value_name="within_cell_z_score",
    )

    long_frame["signal"] = long_frame["signal"].str.removesuffix("_z").map(signal_names)

    sns.lineplot(
        data=long_frame,
        x="assessment_index",
        y="within_cell_z_score",
        hue="signal",
        marker="o",
        markersize=4,
        linewidth=1.8,
        ax=axis,
    )

    challenge_assessments = QC_CHALLENGE_CASES.get(
        cell_id,
        [],
    )

    for assessment_index in challenge_assessments:
        axis.axvline(
            assessment_index,
            color="crimson",
            linestyle=":",
            linewidth=1.5,
            alpha=0.85,
        )

        axis.text(
            assessment_index,
            0.98,
            f"Rejected EIS {assessment_index}",
            transform=axis.get_xaxis_transform(),
            rotation=90,
            verticalalignment="top",
            horizontalalignment="right",
            fontsize=8,
            color="crimson",
        )

        selected_row = cell_frame.loc[cell_frame["assessment_index"] == assessment_index]

        if not selected_row.empty:
            audit_row = all_spectrum_lin_kk.loc[
                (all_spectrum_lin_kk["cell_id"] == cell_id)
                & (all_spectrum_lin_kk["assessment_index"] == assessment_index)
            ]

            record = {
                "cell_id": cell_id,
                "assessment_index": assessment_index,
            }

            for signal in qc_alignment_signals:
                standardized_name = f"{signal}_z"

                record[f"{signal_names[signal]} z-score"] = float(
                    selected_row[standardized_name].iloc[0]
                )

            if not audit_row.empty:
                record["lin_kk_rmse_pct"] = float(audit_row["residual_rmse_pct"].iloc[0])
                record["lin_kk_maximum_residual_pct"] = float(
                    audit_row["maximum_absolute_residual_pct"].iloc[0]
                )
                record["passes_lin_kk"] = bool(audit_row["passes_lin_kk"].iloc[0])

            alignment_records.append(record)

    axis.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    axis.set_title(
        (f"Cell {cell_id}: raw cross-modal context for rejected EIS spectra"),
        fontweight="bold",
    )
    axis.set_xlabel("Degradation assessment index")
    axis.set_ylabel("Within-cell standardized value")
    axis.legend(
        title="Signal",
        fontsize=9,
        loc="best",
    )

fig.suptitle(
    ("Diagnostic alignment of rejected EIS spectra with composite SOH"),
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()

qc_alignment_path = FIGURE_DIRECTORY / "qc_challenge_cross_modal_alignment.png"

fig.savefig(
    qc_alignment_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

qc_challenge_alignment = (
    pd.DataFrame(alignment_records)
    .sort_values(
        [
            "cell_id",
            "assessment_index",
        ]
    )
    .reset_index(drop=True)
)

qc_challenge_alignment.to_csv(
    OUTPUT_DIRECTORY / "qc_challenge_cross_modal_alignment.csv",
    index=False,
)

print("Cross-modal diagnostic values at rejected QC challenge assessments")
display(qc_challenge_alignment.round(4))

print(
    "Diagnostic figure saved to:",
    qc_alignment_path,
)

## 5. Robust within-cell degradation slopes

For each cell, a Theil-Sen slope is fitted against assessment index. It is the median of pairwise slopes and is less sensitive to isolated excursions than ordinary least squares:

$$
\widehat{\beta}_{TS}
=
\operatorname{median}_{i<j}
\left(
\frac{x_j-x_i}{k_j-k_i}
\right)
$$

The slope measures change per assessment, not change per operating hour. Confidence intervals are descriptive because each trajectory is temporally correlated.


In [ ]:
slope_records = []

for cell_id, cell_frame in analysis.groupby("cell_id", observed=True):
    for feature in [*core_relative_features, "soh_composite_pct"]:
        valid = cell_frame[["assessment_index", feature]].dropna()
        if len(valid) < 3:
            continue
        slope, intercept, lower, upper = theilslopes(
            valid[feature].to_numpy(),
            valid["assessment_index"].to_numpy(),
            alpha=0.95,
        )
        slope_records.append(
            {
                "cell_id": cell_id,
                "regime": cell_frame["regime"].iloc[0],
                "feature": feature,
                "observations": len(valid),
                "theil_sen_slope_per_assessment": slope,
                "lower_95_pct": lower,
                "upper_95_pct": upper,
                "interval_excludes_zero": bool(lower > 0 or upper < 0),
            }
        )

slope_results = pd.DataFrame(slope_records)
slope_results.to_csv(
    OUTPUT_DIRECTORY / "within_cell_theil_sen_slopes.csv",
    index=False,
)

display(
    slope_results.pivot_table(
        index=["regime", "cell_id"],
        columns="feature",
        values="theil_sen_slope_per_assessment",
    ).round(3)
)

## 6. Cross-modal association without pseudoreplication

A pooled correlation can be driven by stable differences between cells. We therefore compare:

- Pooled Spearman correlation across all assessments
- Within-cell Spearman correlation after subtracting each cell's mean
- A cell-cluster bootstrap confidence interval that resamples complete cells

For feature $x$:

$$
\widetilde{x}_{c,k}=x_{c,k}-\overline{x}_c
$$

The same transformation is applied to outcome $y$. This estimates whether deviations from a cell's typical impedance align with deviations from its typical performance. It does not by itself prove causality.


In [ ]:
def safe_spearman(x, y) -> float:
    valid = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valid) < 3 or valid["x"].nunique() < 2 or valid["y"].nunique() < 2:
        return np.nan
    return float(spearmanr(valid["x"], valid["y"]).statistic)


def cluster_bootstrap_spearman(
    frame,
    feature,
    outcome,
    repetitions=1000,
    random_state=42,
):
    """Bootstrap complete cells and preserve their repeated measurements."""
    rng = np.random.default_rng(random_state)
    cell_frames = {
        cell_id: group[[feature, outcome]].dropna()
        for cell_id, group in frame.groupby("cell_id", observed=True)
    }
    cell_ids = list(cell_frames)
    estimates = []

    for _ in range(repetitions):
        sampled = rng.choice(cell_ids, size=len(cell_ids), replace=True)
        blocks = []
        for bootstrap_id, cell_id in enumerate(sampled):
            block = cell_frames[cell_id].copy()
            block["bootstrap_cell"] = bootstrap_id
            blocks.append(block)
        sample = pd.concat(blocks, ignore_index=True)
        sample["feature_centered"] = sample[feature] - sample.groupby("bootstrap_cell")[
            feature
        ].transform("mean")
        sample["outcome_centered"] = sample[outcome] - sample.groupby("bootstrap_cell")[
            outcome
        ].transform("mean")
        estimate = safe_spearman(
            sample["feature_centered"],
            sample["outcome_centered"],
        )
        if np.isfinite(estimate):
            estimates.append(estimate)

    if not estimates:
        return np.nan, np.nan
    return tuple(np.quantile(estimates, [0.025, 0.975]))


association_records = []
for feature in EIS_FEATURES:
    feature_centered = analysis[feature] - analysis.groupby("cell_id", observed=True)[
        feature
    ].transform("mean")

    for outcome in OUTCOMES:
        outcome_centered = analysis[outcome] - analysis.groupby("cell_id", observed=True)[
            outcome
        ].transform("mean")
        lower, upper = cluster_bootstrap_spearman(
            analysis,
            feature,
            outcome,
        )
        association_records.append(
            {
                "feature": feature,
                "outcome": outcome,
                "observations": int(analysis[[feature, outcome]].dropna().shape[0]),
                "cells": int(analysis.dropna(subset=[feature, outcome])["cell_id"].nunique()),
                "pooled_spearman": safe_spearman(
                    analysis[feature],
                    analysis[outcome],
                ),
                "within_cell_spearman": safe_spearman(
                    feature_centered,
                    outcome_centered,
                ),
                "cluster_bootstrap_lower_95_pct": lower,
                "cluster_bootstrap_upper_95_pct": upper,
                "robust_direction": bool(
                    np.isfinite(lower) and np.isfinite(upper) and (lower > 0 or upper < 0)
                ),
            }
        )

association_results = pd.DataFrame(association_records)
association_results["absolute_within_cell_spearman"] = association_results[
    "within_cell_spearman"
].abs()
association_results.to_csv(
    OUTPUT_DIRECTORY / "eis_cross_modal_associations.csv",
    index=False,
)

print("Strongest within-cell associations")
display(
    association_results.sort_values(
        "absolute_within_cell_spearman",
        ascending=False,
    )
    .head(15)
    .round(3)
)

In [ ]:
core_associations = association_results.loc[
    association_results["feature"].isin(CORE_EIS_FEATURES)
].copy()
correlation_matrix = core_associations.pivot(
    index="feature",
    columns="outcome",
    values="within_cell_spearman",
).reindex(CORE_EIS_FEATURES)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    center=0,
    vmin=-1,
    vmax=1,
    cmap="vlag",
    cbar_kws={"label": "Within-cell Spearman correlation"},
    ax=ax,
)
ax.set_title(
    "Within-cell coupling between EIS and performance",
    fontweight="bold",
)
ax.set_xlabel("Performance or health outcome")
ax.set_ylabel("EIS indicator")

association_figure_path = FIGURE_DIRECTORY / "eis_health_correlation_heatmap.png"
plt.savefig(association_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", association_figure_path)

## 7. Regime-specific coupling

The regular and randomized-redox cells showed different forecast behaviour in Notebooks 06 and 07. A pooled association is therefore accepted only if its direction is reasonably stable across regimes.

Only two randomized-redox cells are available. Their estimates are diagnostic and cannot support a population-level claim. A sign reversal between regimes is treated as evidence against a universal relationship.


In [ ]:
regime_records = []

for regime, regime_frame in analysis.groupby("regime", observed=True):
    for feature in CORE_EIS_FEATURES:
        feature_centered = regime_frame[feature] - regime_frame.groupby("cell_id", observed=True)[
            feature
        ].transform("mean")

        for outcome in OUTCOMES:
            outcome_centered = regime_frame[outcome] - regime_frame.groupby(
                "cell_id", observed=True
            )[outcome].transform("mean")
            regime_records.append(
                {
                    "regime": regime,
                    "feature": feature,
                    "outcome": outcome,
                    "cells": regime_frame.dropna(subset=[feature, outcome])["cell_id"].nunique(),
                    "observations": regime_frame[[feature, outcome]].dropna().shape[0],
                    "within_cell_spearman": safe_spearman(
                        feature_centered,
                        outcome_centered,
                    ),
                }
            )

regime_associations = pd.DataFrame(regime_records)
regime_comparison = regime_associations.pivot_table(
    index=["feature", "outcome"],
    columns="regime",
    values="within_cell_spearman",
).reset_index()
if {"regular", "randomized"}.issubset(regime_comparison.columns):
    regime_comparison["sign_reversal"] = np.sign(regime_comparison["regular"]) != np.sign(
        regime_comparison["randomized"]
    )
    regime_comparison["absolute_regime_difference"] = (
        regime_comparison["regular"] - regime_comparison["randomized"]
    ).abs()

regime_comparison.to_csv(
    OUTPUT_DIRECTORY / "regime_specific_eis_associations.csv",
    index=False,
)
display(regime_comparison.round(3))

## 8. Does EIS add predictive information on an unseen cell?

Association is not enough. We now compare three direct forecasts of future SOH change:

1. Persistence: predicted change is zero.
2. History model: current SOH, assessment index and recent SOH change.
3. EIS-augmented model: the history predictors plus current and recent changes in baseline-relative ohmic and polarization resistance.

For horizon $h$:

$$
\Delta SOH_{c,k,h}=SOH_{c,k+h}-SOH_{c,k}
$$

The outer fold holds out a complete cell. Ridge regularization is selected inside the remaining cells using inner leave-one-cell-out validation and equal weighting across inner validation cells.


In [ ]:
HORIZONS = [1, 3, 5, 10]
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]

# Forecast origins must have accepted EIS spectra.
predictive = analysis.copy()

predictive["soh_at_origin_pct"] = predictive["soh_composite_pct"]

# Obtain the previous SOH from the complete modeling table.
# A rejected EIS spectrum does not invalidate its SOH measurement.
previous_soh_lookup = modeling_all[
    [
        "cell_id",
        "assessment_index",
        "soh_composite_pct",
    ]
].rename(
    columns={
        "assessment_index": ("previous_assessment_index"),
        "soh_composite_pct": ("previous_soh_pct"),
    }
)

predictive["previous_assessment_index"] = predictive["assessment_index"] - 1

predictive = predictive.merge(
    previous_soh_lookup,
    on=[
        "cell_id",
        "previous_assessment_index",
    ],
    how="left",
    validate="many_to_one",
)

predictive["soh_delta1"] = predictive["soh_at_origin_pct"] - predictive["previous_soh_pct"]

# EIS changes require both the current and previous spectra
# to have passed the Lin-KK acceptance screen.
previous_eis_lookup = analysis[
    [
        "cell_id",
        "assessment_index",
        *core_relative_features,
    ]
].copy()

previous_eis_lookup = previous_eis_lookup.rename(
    columns={
        "assessment_index": ("previous_assessment_index"),
        **{feature: f"{feature}_previous" for feature in core_relative_features},
    }
)

predictive = predictive.merge(
    previous_eis_lookup,
    on=[
        "cell_id",
        "previous_assessment_index",
    ],
    how="left",
    validate="many_to_one",
)

for feature in core_relative_features:
    predictive[f"{feature}_delta1"] = predictive[feature] - predictive[f"{feature}_previous"]

HISTORY_PREDICTORS = [
    "soh_at_origin_pct",
    "assessment_index",
    "soh_delta1",
]

EIS_PREDICTORS = [
    *HISTORY_PREDICTORS,
    *core_relative_features,
    *[f"{feature}_delta1" for feature in core_relative_features],
]

# Future SOH targets come from the complete modeling table.
# Only the origin EIS spectrum is required to pass Lin-KK.
future_lookup = modeling_all[
    [
        "cell_id",
        "assessment_index",
        "soh_composite_pct",
    ]
].rename(
    columns={
        "assessment_index": ("future_assessment_index"),
        "soh_composite_pct": ("future_soh_pct"),
    }
)

forecast_frames = []

for horizon in HORIZONS:
    frame = predictive.copy()

    frame["horizon"] = horizon

    frame["future_assessment_index"] = frame["assessment_index"] + horizon

    frame = frame.merge(
        future_lookup,
        on=[
            "cell_id",
            "future_assessment_index",
        ],
        how="inner",
        validate="many_to_one",
    )

    frame["delta_soh_pct"] = frame["future_soh_pct"] - frame["soh_at_origin_pct"]

    if not (frame["future_assessment_index"] - frame["assessment_index"]).eq(horizon).all():
        raise ValueError("Forecast horizon alignment failed.")

    forecast_frames.append(frame)

forecast_data = pd.concat(
    forecast_frames,
    ignore_index=True,
)

print("Forecast rows:", len(forecast_data))
print(
    "Accepted EIS origin spectra:",
    predictive[
        [
            "cell_id",
            "assessment_index",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "History predictors:",
    HISTORY_PREDICTORS,
)
print(
    "EIS-augmented predictors:",
    EIS_PREDICTORS,
)

display(
    forecast_data.groupby(
        "horizon",
        observed=True,
    ).agg(
        rows=("delta_soh_pct", "size"),
        cells=("cell_id", "nunique"),
        first_origin=("assessment_index", "min"),
        last_origin=("assessment_index", "max"),
    )
)

In [ ]:
def make_ridge(alpha):
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", RobustScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def select_alpha_by_inner_cells(training, predictors):
    records = []
    inner_cells = sorted(training["cell_id"].unique())

    for alpha in RIDGE_ALPHAS:
        cell_maes = []
        for validation_cell in inner_cells:
            inner_train = training.loc[training["cell_id"] != validation_cell]
            inner_validation = training.loc[training["cell_id"] == validation_cell]
            estimator = make_ridge(alpha)
            estimator.fit(
                inner_train[predictors],
                inner_train["delta_soh_pct"],
            )
            prediction = estimator.predict(inner_validation[predictors])
            cell_maes.append(
                mean_absolute_error(
                    inner_validation["delta_soh_pct"],
                    prediction,
                )
            )
        records.append(
            {
                "alpha": alpha,
                "inner_macro_cell_mae": float(np.mean(cell_maes)),
                "inner_worst_cell_mae": float(np.max(cell_maes)),
            }
        )

    tuning = pd.DataFrame(records).sort_values(
        ["inner_macro_cell_mae", "inner_worst_cell_mae", "alpha"]
    )
    return float(tuning.iloc[0]["alpha"]), tuning


metric_records = []
prediction_records = []
tuning_frames = []

for horizon in HORIZONS:
    horizon_data = forecast_data.loc[forecast_data["horizon"] == horizon]

    for held_out_cell in sorted(horizon_data["cell_id"].unique()):
        train = horizon_data.loc[horizon_data["cell_id"] != held_out_cell]
        test = horizon_data.loc[horizon_data["cell_id"] == held_out_cell]
        if held_out_cell in train["cell_id"].unique():
            raise ValueError("Outer cell leakage detected.")

        actual_delta = test["delta_soh_pct"].to_numpy()
        persistence_delta = np.zeros(len(test))
        persistence_mae = mean_absolute_error(actual_delta, persistence_delta)

        for model_name, predictors in {
            "history_ridge": HISTORY_PREDICTORS,
            "eis_augmented_ridge": EIS_PREDICTORS,
        }.items():
            best_alpha, tuning = select_alpha_by_inner_cells(train, predictors)
            tuning = tuning.assign(
                horizon=horizon,
                held_out_cell=held_out_cell,
                model=model_name,
            )
            tuning_frames.append(tuning)

            estimator = make_ridge(best_alpha)
            estimator.fit(train[predictors], train["delta_soh_pct"])
            predicted_delta = estimator.predict(test[predictors])
            model_mae = mean_absolute_error(actual_delta, predicted_delta)

            metric_records.append(
                {
                    "horizon": horizon,
                    "cell_id": held_out_cell,
                    "regime": test["regime"].iloc[0],
                    "model": model_name,
                    "best_alpha": best_alpha,
                    "mae": model_mae,
                    "persistence_mae": persistence_mae,
                    "skill_vs_persistence": (
                        1 - model_mae / persistence_mae if persistence_mae > 0 else np.nan
                    ),
                }
            )

            for row_index, (_, row) in enumerate(test.iterrows()):
                prediction_records.append(
                    {
                        "horizon": horizon,
                        "cell_id": held_out_cell,
                        "regime": row["regime"],
                        "assessment_index": row["assessment_index"],
                        "future_assessment_index": row["future_assessment_index"],
                        "model": model_name,
                        "actual_delta_soh_pct": actual_delta[row_index],
                        "predicted_delta_soh_pct": predicted_delta[row_index],
                    }
                )

eis_predictive_metrics = pd.DataFrame(metric_records)
eis_predictive_predictions = pd.DataFrame(prediction_records)
eis_predictive_tuning = pd.concat(tuning_frames, ignore_index=True)

eis_predictive_metrics.to_csv(
    OUTPUT_DIRECTORY / "eis_incremental_predictive_metrics.csv",
    index=False,
)
eis_predictive_predictions.to_csv(
    OUTPUT_DIRECTORY / "eis_incremental_predictions.csv",
    index=False,
)
eis_predictive_tuning.to_csv(
    OUTPUT_DIRECTORY / "eis_incremental_tuning.csv",
    index=False,
)

# Independently audit every outer and inner cell split.
leakage_audit_records = []

for horizon in HORIZONS:
    horizon_data = forecast_data.loc[forecast_data["horizon"] == horizon]

    outer_cells = sorted(horizon_data["cell_id"].unique())

    for held_out_cell in outer_cells:
        outer_train = horizon_data.loc[horizon_data["cell_id"] != held_out_cell]

        outer_test = horizon_data.loc[horizon_data["cell_id"] == held_out_cell]

        outer_train_cells = set(outer_train["cell_id"].unique())
        outer_test_cells = set(outer_test["cell_id"].unique())

        outer_overlap = outer_train_cells.intersection(outer_test_cells)

        leakage_audit_records.append(
            {
                "horizon": horizon,
                "outer_held_out_cell": (held_out_cell),
                "split_level": "outer",
                "inner_validation_cell": None,
                "training_cells": ",".join(sorted(outer_train_cells)),
                "validation_or_test_cells": (",".join(sorted(outer_test_cells))),
                "overlapping_cells": ",".join(sorted(outer_overlap)),
                "leakage_violation": bool(outer_overlap),
            }
        )

        inner_cells = sorted(outer_train["cell_id"].unique())

        for validation_cell in inner_cells:
            inner_train = outer_train.loc[outer_train["cell_id"] != validation_cell]

            inner_validation = outer_train.loc[outer_train["cell_id"] == validation_cell]

            inner_train_cells = set(inner_train["cell_id"].unique())
            inner_validation_cells = set(inner_validation["cell_id"].unique())

            inner_overlap = inner_train_cells.intersection(inner_validation_cells)

            # Also verify that the outer held-out cell
            # never enters inner training or validation.
            outer_cell_reentry = (
                held_out_cell in inner_train_cells or held_out_cell in inner_validation_cells
            )

            leakage_audit_records.append(
                {
                    "horizon": horizon,
                    "outer_held_out_cell": (held_out_cell),
                    "split_level": "inner",
                    "inner_validation_cell": (validation_cell),
                    "training_cells": ",".join(sorted(inner_train_cells)),
                    "validation_or_test_cells": (",".join(sorted(inner_validation_cells))),
                    "overlapping_cells": (",".join(sorted(inner_overlap))),
                    "leakage_violation": bool(inner_overlap or outer_cell_reentry),
                }
            )

cell_leakage_audit = pd.DataFrame(leakage_audit_records)

cell_leakage_violations = int(cell_leakage_audit["leakage_violation"].sum())

leakage_summary = (
    cell_leakage_audit.groupby(
        "split_level",
        observed=True,
    )
    .agg(
        splits_audited=(
            "leakage_violation",
            "size",
        ),
        leakage_violations=(
            "leakage_violation",
            "sum",
        ),
    )
    .reset_index()
)

cell_leakage_audit.to_csv(
    OUTPUT_DIRECTORY / "eis_forecast_cell_leakage_audit.csv",
    index=False,
)

print(
    "Outer metric rows:",
    len(eis_predictive_metrics),
)
print(
    "Outer prediction rows:",
    len(eis_predictive_predictions),
)
print(
    "Cell-leakage violations:",
    cell_leakage_violations,
)

display(leakage_summary)

assert cell_leakage_violations == 0, f"Detected {cell_leakage_violations} cell-leakage violations."

In [ ]:
model_summary = (
    eis_predictive_metrics.groupby(["horizon", "model"], observed=True)
    .agg(
        cells=("cell_id", "nunique"),
        macro_cell_mae=("mae", "mean"),
        persistence_macro_mae=("persistence_mae", "mean"),
        median_cell_skill=("skill_vs_persistence", "median"),
        minimum_cell_skill=("skill_vs_persistence", "min"),
        cells_beating_persistence=(
            "skill_vs_persistence",
            lambda x: int((x > 0).sum()),
        ),
    )
    .reset_index()
)
model_summary["macro_skill_vs_persistence"] = 1 - (
    model_summary["macro_cell_mae"] / model_summary["persistence_macro_mae"]
)

incremental_comparison = eis_predictive_metrics.pivot_table(
    index=["horizon", "cell_id", "regime"],
    columns="model",
    values="mae",
).reset_index()
incremental_comparison["eis_mae_improvement"] = (
    incremental_comparison["history_ridge"] - incremental_comparison["eis_augmented_ridge"]
)
incremental_comparison["eis_relative_improvement"] = (
    incremental_comparison["eis_mae_improvement"] / incremental_comparison["history_ridge"]
)

incremental_summary = (
    incremental_comparison.groupby("horizon", observed=True)
    .agg(
        cells=("cell_id", "nunique"),
        median_relative_improvement=("eis_relative_improvement", "median"),
        mean_relative_improvement=("eis_relative_improvement", "mean"),
        minimum_relative_improvement=("eis_relative_improvement", "min"),
        cells_improved=(
            "eis_relative_improvement",
            lambda x: int((x > 0).sum()),
        ),
    )
    .reset_index()
)

model_summary.to_csv(
    OUTPUT_DIRECTORY / "eis_predictive_horizon_summary.csv",
    index=False,
)
incremental_comparison.to_csv(
    OUTPUT_DIRECTORY / "eis_incremental_cell_comparison.csv",
    index=False,
)
incremental_summary.to_csv(
    OUTPUT_DIRECTORY / "eis_incremental_horizon_summary.csv",
    index=False,
)

print("Predictive performance by horizon")
display(model_summary.round(3))
print("Incremental value of EIS beyond health history")
display(incremental_summary.round(3))

In [ ]:
skill_matrix = eis_predictive_metrics.loc[
    eis_predictive_metrics["model"] == "eis_augmented_ridge"
].pivot(index="cell_id", columns="horizon", values="skill_vs_persistence")
skill_matrix = skill_matrix.reindex(
    index=[
        cell
        for cell in ["N1", "N2", "N3", "N4", "N5", "N6", "R1", "R2"]
        if cell in skill_matrix.index
    ],
    columns=HORIZONS,
)

increment_matrix = incremental_comparison.pivot(
    index="cell_id",
    columns="horizon",
    values="eis_relative_improvement",
).reindex(index=skill_matrix.index, columns=HORIZONS)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
sns.heatmap(
    skill_matrix,
    annot=True,
    fmt="+.2f",
    center=0,
    cmap="RdYlGn",
    cbar_kws={"label": "MAE skill versus persistence"},
    ax=axes[0],
)
axes[0].set_title("EIS-augmented skill", fontweight="bold")
axes[0].set_xlabel("Forecast horizon")
axes[0].set_ylabel("Held-out physical cell")

sns.heatmap(
    increment_matrix,
    annot=True,
    fmt="+.2f",
    center=0,
    cmap="RdYlGn",
    cbar_kws={"label": "Relative MAE improvement"},
    ax=axes[1],
)
axes[1].set_title("EIS value beyond health history", fontweight="bold")
axes[1].set_xlabel("Forecast horizon")
axes[1].set_ylabel("Held-out physical cell")

fig.suptitle(
    "Leakage-safe cross-cell predictive value of EIS",
    fontsize=17,
    fontweight="bold",
)
plt.tight_layout()
predictive_figure_path = FIGURE_DIRECTORY / "eis_incremental_prediction_skill.png"
plt.savefig(predictive_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", predictive_figure_path)

## 9. Engineering decision logic

The final register separates five questions that must not be conflated: Is the spectrum structurally complete? Is it physically consistent? Does EIS track health? Does it add unseen-cell predictive value? Is the result ready for deployment?


In [ ]:
robust_associations = int(association_results["robust_direction"].sum())
sign_reversals = int(regime_comparison["sign_reversal"].sum())
positive_incremental_horizons = int((incremental_summary["median_relative_improvement"] > 0).sum())
all_kk_passes = int(all_spectrum_lin_kk["passes_lin_kk"].sum())
all_kk_failures = int((~all_spectrum_lin_kk["passes_lin_kk"]).sum())
package_fit_failures = int((circuit_comparison["status"] != "success").sum())
numerical_package_fits = int((circuit_comparison["status"] == "success").sum())
parameter_uncertainty = ecm_parameters.groupby("label", observed=True)["Std. err. (%)"].max()
non_identifiable_labels = sorted(
    parameter_uncertainty.loc[
        (~np.isfinite(parameter_uncertainty)) | parameter_uncertainty.gt(100)
    ].index
)
largest_regime_difference = float(regime_comparison["absolute_regime_difference"].max())
one_arc_rmse = circuit_comparison.loc[
    circuit_comparison["status"] == "success",
    "relative_complex_rmse_pct",
]

decision_register = pd.DataFrame(
    [
        {
            "decision": "Raw-spectrum structural consistency",
            "evidence": (
                f"{len(spectral_audit)} spectra share the acquisition grid; "
                f"{int((~spectral_audit['passes_basic_qc']).sum())} basic-QC failures; "
                f"{int(continuity_audit['continuity_review_required'].sum())} "
                "frequency-aware continuity flags"
            ),
            "status": "STRUCTURALLY SUPPORTED; CONTINUITY REVIEW RECORDED",
        },
        {
            "decision": "All-spectrum Lin-KK validation",
            "evidence": (
                f"{all_kk_passes} of {len(all_spectrum_lin_kk)} spectra passed both "
                f"RMSE <= {LIN_KK_RMSE_LIMIT_PCT:.1f}% and maximum residual "
                f"<= {LIN_KK_MAX_RESIDUAL_LIMIT_PCT:.1f}%; "
                f"{all_kk_failures} excluded from downstream EIS interpretation"
            ),
            "status": "FULL DATASET SCREENED",
        },
        {
            "decision": "One-arc circuit adequacy",
            "evidence": (
                f"{numerical_package_fits} package fits completed numerically with median "
                f"relative complex RMSE={one_arc_rmse.median():.2f}%; "
                f"{len(non_identifiable_labels)} pyimpspec spectrum has parameter "
                f"uncertainty above 100%; {package_fit_failures} exceptions; "
                "DRT resolves multiple relaxation regions"
            ),
            "status": "RESISTANCE SENSITIVITY REPORTED; UNIQUE MECHANISM NOT SUPPORTED",
        },
        {
            "decision": "Cross-modal association",
            "evidence": (
                f"{robust_associations} feature-outcome cluster-bootstrap intervals "
                "exclude zero; correlated features and shared time trends remain"
            ),
            "status": "EXPLORATORY WITH EIGHT CELLS",
        },
        {
            "decision": "Regime stability",
            "evidence": (
                f"{sign_reversals} algebraic sign reversals, several near zero; "
                f"largest absolute regime difference={largest_regime_difference:.3f}; "
                "randomized-redox evidence comes from two cells"
            ),
            "status": "REGIME DEPENDENCE DETECTED; LIMITED REPLICATION",
        },
        {
            "decision": "Incremental predictive value",
            "evidence": (
                f"positive median EIS improvement at {positive_incremental_horizons} "
                f"of {len(HORIZONS)} horizons"
            ),
            "status": "NOT SUPPORTED BEYOND HEALTH HISTORY",
        },
        {
            "decision": "Mechanism claim",
            "evidence": (
                "EIS, IV and transient observations align by assessment, but the "
                "experiment does not identify unique causal mechanisms"
            ),
            "status": "ASSOCIATION, NOT CAUSATION",
        },
        {
            "decision": "Cell-isolated validation",
            "evidence": (
                f"{len(cell_leakage_audit)} inner and outer splits audited; "
                f"{cell_leakage_violations} physical-cell overlap violations"
            ),
            "status": "LEAKAGE AUDIT PASSED",
        },
        {
            "decision": "Deployment",
            "evidence": ("Eight laboratory cells, no operating-hour axis and no field telemetry"),
            "status": "NOT VALIDATED",
        },
    ]
)
decision_register.to_csv(
    OUTPUT_DIRECTORY / "notebook08_decision_register.csv",
    index=False,
)
print("Notebook 08 decision register")
display(decision_register)

## Final engineering decision

The EIS dataset is structurally consistent, with all 418 spectra sharing the same 61-point frequency grid. Full Lin-KK screening accepted 407 spectra and rejected 11 spectra using the project-specific limits of residual RMSE ≤ 1% and maximum absolute residual ≤ 5%. The rejected measurements are retained for traceability but excluded from EIS interpretation and predictive modelling.

For Lin-KK-accepted spectra, polarization resistance is the most informative EIS degradation indicator. It has strong negative within-cell associations with composite SOH, IV current, maximum power and transient current. Ohmic resistance shows comparatively weak and inconsistent relationships with performance.

The polarization-health relationship is strong for regularly operated cells but weak for randomized-redox cells. This indicates regime-dependent EIS behaviour. However, the randomized-redox evidence is based on only two physical cells and should therefore be considered exploratory.

DRT identifies multiple relaxation regions in the representative spectra. Consequently, the single-arc `R(RC)` equivalent circuit is structurally too simple for unique mechanistic interpretation. Resistance estimates are moderately stable across `pyimpspec` and `impedance.py`, but capacitance and the N1 assessment-1 circuit parameters are not reliably identifiable.

The leakage-safe cross-cell forecasting experiment shows that SOH history consistently outperforms persistence, particularly at longer horizons. Adding EIS predictors does not provide positive median improvement beyond SOH history at any evaluated horizon. EIS should therefore be retained as a diagnostic and explanatory measurement, but not accepted as a generally useful incremental forecasting input for this dataset.

All 256 outer and inner validation splits were explicitly audited, with zero physical-cell overlap violations. Therefore, the results support an offline laboratory diagnostic conclusion. They do not support a unique causal degradation mechanism, operating-hour RUL estimation or field deployment.

## Reproducibility outputs

Tables are saved under `reports/tables/eis_health_coupling/`:

- `spectral_quality_audit.csv`
- `spectral_coverage_by_cell.csv`
- `eis_continuity_audit.csv`
- `eis_continuity_by_cell.csv`
- `all_spectrum_lin_kk_audit.csv`
- `lin_kk_pass_fail_by_cell.csv`
- `failed_lin_kk_spectra.csv`
- `pyimpspec_kramers_kronig_summary.csv`
- `pyimpspec_ecm_parameters.csv`
- `pyimpspec_ecm_statistics.csv`
- `pyimpspec_drt_summary.csv`
- `cross_package_circuit_fits.csv`
- `cross_package_parameter_sensitivity.csv`
- `qc_challenge_lin_kk_summary.csv`
- `qc_challenge_cross_modal_alignment.csv`
- `baseline_relative_eis_summary.csv`
- `within_cell_theil_sen_slopes.csv`
- `eis_cross_modal_associations.csv`
- `regime_specific_eis_associations.csv`
- `eis_incremental_predictive_metrics.csv`
- `eis_incremental_predictions.csv`
- `eis_incremental_tuning.csv`
- `eis_forecast_cell_leakage_audit.csv`
- `eis_predictive_horizon_summary.csv`
- `eis_incremental_cell_comparison.csv`
- `eis_incremental_horizon_summary.csv`
- `notebook08_decision_register.csv`

Figures are saved under `reports/figures/eis_health_coupling/`.

The notebook is complete when **Restart and Run All** produces every listed artifact, audits 32 outer and 224 inner splits with zero cell-leakage violations, screens all 418 spectra against both declared Lin-KK limits, and ends with a decision register that distinguishes structural quality, association, incremental prediction, mechanistic interpretation and deployment.
